# Visual ML Foundations: Test-Time Augmentation (TTA) Mechanics
### Squeezing $0.005$ to $0.015$ Metric Gains on the Leaderboard

---

## 1. Concept Overview
During inference on test images, single predictions are vulnerable to slight pixel noise.
**Test-Time Augmentation (TTA)**:
1. Feeds the original image plus subtle geometric variants (Horizontal Flip, Slight Scale).
2. Computes model predictions on each variant.
3. Inverts the transformation (if spatial) and averages the predicted probability distributions:
   $$\bar{p} = \frac{1}{K} \sum_{k=1}^K p(T_k(X))$$
This dramatically stabilizes model predictions on difficult borderline cases!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate knee image
np.random.seed(42)
img = np.zeros((100, 100))
y, x = np.ogrid[:100, :100]
img[((y - 40)**2 + (x - 50)**2) < 22**2] = 0.8
img[((y - 75)**2 + (x - 50)**2) < 20**2] = 0.7

# Create 4 TTA variants
orig = img
flip_h = np.fliplr(img)
shift_pos = np.roll(img, shift=4, axis=1)
contrast_boost = np.clip(img * 1.25, 0, 1)

tta_variants = [orig, flip_h, shift_pos, contrast_boost]
titles = ["Original", "Horizontal Flip", "Horizontal Shift (+4px)", "Contrast Shift (1.25x)"]
simulated_preds = [0.912, 0.884, 0.905, 0.931]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, t, p in zip(axes, tta_variants, titles, simulated_preds):
    ax.imshow(im, cmap='bone')
    ax.set_title(f"{t}\nPred: {p:.3f}", fontsize=11, fontweight='bold')
    ax.axis('off')

mean_p = np.mean(simulated_preds)
plt.suptitle(f"Test-Time Augmentation (TTA) Ensemble | Calibrated Prediction: {mean_p:.4f}", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"TTA Ensembled Probability: {mean_p:.4f} (Variance reduced across transforms)")
